# Scrape Florida House Member IDs
9 August 2024

on myfloridahouse.gov, each state legislator has a corresponding photo
e.g. https://www.myfloridahouse.gov/FileStores/Web/Imaging/Member/4876.jpg

Each legislator has a member ID, which could be scraped to identify their corresponding photos, at
https://www.myfloridahouse.gov/Sections/Representatives/representatives.aspx

This notebook creates a dataframe with the following fields:
* district_number
* legislator_name
* mfh_member_id

### Technical Notes
Look at https://www.myfloridahouse.gov/robots.txt to see how scrapers might be blocked.

Scraping via the *requests* library did not work. Received a 200 response with the body, "The requested URL was rejected. Please consult with your administrator."

To work around this, I'm using Playwright, a user-interaction emulator. Jonathan Soma (Columbia University) has a [Playwright tutorial](https://jsoma.github.io/advanced-scraping-with-playwright/)  that proved super helpful here.

I had problems extracting the image source url after the first row:
print(legislator.find('img').get('src'))
So I instead extracted the member ID from the legislator page url

In [13]:
# import requests

In [2]:
# url = "https://www.myfloridahouse.gov/Sections/Representatives/representatives.aspx"
# headers = {
#     "User-Agent": "MyScrapingScript/1.0 (+https://https://jaxtrib.org/)",
#     "From": "rob.reid@jaxtrib.org",
#     "Purpose": "Retrieve urls for legislator photos for use with a public interest legislative voting patterns dashboard"
# }
# response = requests.get(url)
# html = response.text
# html

# 1) Set Up

In [ ]:
#!pip install playwright
#!playwright install
#!sudo apt-get install libnss3 libnspr4 libgbm1 libasound2

In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import re #regex, for extracting memberID
from playwright.async_api import async_playwright

### Instantiate Browser

In [3]:
playwright = await async_playwright().start()

In [7]:
browser = await playwright.chromium.launch(headless=False)

In [8]:
# Create a new browser window
page = await browser.new_page()

# 2) Retrieve Legislator Webpage and Assember Data Frame

In [9]:
# Tell it to go to this page
await page.goto("https://www.myfloridahouse.gov/Sections/Representatives/representatives.aspx")

<Response url='https://www.myfloridahouse.gov/Sections/Representatives/representatives.aspx' request=<Request url='https://www.myfloridahouse.gov/Sections/Representatives/representatives.aspx' method='GET'>>

In [10]:
html_content = await page.content()
doc = BeautifulSoup(html_content)

In [11]:
legislators = doc.find_all(class_='team-box')

### a) find relevant info

In [29]:

for legislator in legislators:
    print("----")
    print(legislator.find('h5').text.strip())
    print(legislator.find(class_='text-nowrap').text)
    print(legislator.find('a').get('href'))

----
Abbott, Shane G.
District: 5
/Sections/Representatives/details.aspx?MemberId=4864&LegislativeTermId=90
----
Altman, Thad
District: 32
/Sections/Representatives/details.aspx?MemberId=4329&LegislativeTermId=90
----
Alvarez, Daniel Antonio "Danny"
District: 69
/Sections/Representatives/details.aspx?MemberId=4889&LegislativeTermId=90
----
Amesty, Carolina
District: 45
/Sections/Representatives/details.aspx?MemberId=4876&LegislativeTermId=90
----
Anderson, Adam
District: 57
/Sections/Representatives/details.aspx?MemberId=4885&LegislativeTermId=90
----
Andrade, Robert Alexander "Alex"
District: 2
/Sections/Representatives/details.aspx?MemberId=4710&LegislativeTermId=90
----
Antone, Bruce Hadley
District: 41
/Sections/Representatives/details.aspx?MemberId=4269&LegislativeTermId=90
----
Arrington, Kristen Aston
District: 46
/Sections/Representatives/details.aspx?MemberId=4774&LegislativeTermId=90
----
Baker, Jessica
District: 17
/Sections/Representatives/details.aspx?MemberId=4866&Legisla

### b) extract info

In [43]:
for legislator in legislators:
    print('-----')
    print(legislator.find('h5').text.strip().split(',')[0])
    print(legislator.find(class_='text-nowrap').text.split(':')[1].strip())
    # if legislator.find('img').get('src'):
    #     print(re.search(r'Member/(\d+)\.jpg', legislator.find('img').get('src')).group(1))
    if legislator.find('a').get('href'):
        match = re.search(r'MemberId=(\d+)\&Leg', legislator.find('a').get('href'))
        if match: print(match.group(1))

-----
Abbott
5
4864
-----
Altman
32
4329
-----
Alvarez
69
4889
-----
Amesty
45
4876
-----
Anderson
57
4885
-----
Andrade
2
4710
-----
Antone
41
4269
-----
Arrington
46
4774
-----
Baker
17
4866
-----
Bankson
39
4873
-----
Barnaby
29
4770
-----
Bartleman
103
4793
-----
Basabe
106
4900
-----
Bell
49
4753
-----
Beltran
70
4747
-----
Benjamin
107
4795
-----
Berfield
58
4210
-----
Black
15
4865
-----
Borrero
111
4794
-----
Botana
80
4782
-----
Brackett
34
4870
-----
Bracy Davis
40
4874
-----
Brannan III
10
4708
-----
Buchanan
74
4731
-----
Busatta Cabrera
114
4800
-----
Campbell
99
4843
-----
Canady
50
4879
-----
Caruso
87
4754
-----
Casello
90
4750
-----
Cassel
101
4895
-----
Chamberlin
24
4901
-----
Chambliss
117
4798
-----
Chaney
61
4778
-----
Clemons
22
4655
-----
Cross
60
4887
-----
Daley
96
4757
-----
Daniels
14
4652
-----
Driskell
67
4745
-----
Duggan
12
4712
-----
Dunkley
97
4893
-----
Edmonds
88
4842
-----
Eskamani
42
4746
-----
Esposito
77
4892
-----
Fabricio
110
4792
-----
Fine
33

### c) store info in data frame

In [44]:
all_data = []
for legislator in legislators:
    data = {
        'last_name':legislator.find('h5').text.strip().split(',')[0],
        'district_number':legislator.find(class_='text-nowrap').text.split(':')[1].strip(),
        'mfh_member_id':re.search(r'MemberId=(\d+)\&Leg', legislator.find('a').get('href')).group(1)
            if legislator.find('a').get('href') else None
    }
    all_data.append(data)

In [45]:
df = pd.DataFrame(all_data)
df

,last_name,district_number,mfh_member_id
0,Abbott,5,4864
1,Altman,32,4329
2,Alvarez,69,4889
3,Amesty,45,4876
4,Anderson,57,4885
...,...,...,...
119,Yeager,56,4882
120,Fernandez-Barquin,118,4709
121,Gregory,72,4732
122,Harding,24,4769


# 3) Save Results

In [46]:
df.to_csv("../data-raw/myfloridahouse/t_myfloridahouse_member_ids.csv")